In [10]:
import os
import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine

load_dotenv()
engine = create_engine(os.getenv("DATABASE_URL"))

df = pd.read_sql("SELECT * FROM machine_readings;", engine)
df.head()

,udi,product_id,type,air_temperature_k,process_temperature_k,rotational_speed_rpm,torque_nm,tool_wear_min,machine_failure,twf,hdf,pwf,osf,rnf,loaded_at
0,19,H29432,H,298.8,309.2,1306,54.5,50,0,0,0,0,0,0,2026-08-22 16:02:51.767860
1,20,M14879,M,298.9,309.3,1632,32.5,55,0,0,0,0,0,0,2026-08-22 16:02:51.767860
2,21,H29434,H,298.9,309.3,1375,42.7,58,0,0,0,0,0,0,2026-08-22 16:02:51.767860
3,22,L47201,L,298.8,309.3,1450,44.8,63,0,0,0,0,0,0,2026-08-22 16:02:51.767860
4,23,M14882,M,298.9,309.3,1581,30.7,65,0,0,0,0,0,0,2026-08-22 16:02:51.767860


In [11]:
df['machine_failure'].value_counts()

machine_failure
0    9661
1     339
Name: count, dtype: int64

In [3]:
machine_failure
0    9661
1     339
Name: count, dtype: int64

SyntaxError: invalid syntax (1628615088.py, line 2)

In [12]:
df.groupby('type')['machine_failure'].mean().sort_values(ascending=False)


type
L    0.039167
M    0.027694
H    0.020937
Name: machine_failure, dtype: float64

In [13]:
df.groupby(pd.cut(df['tool_wear_min'], bins=10))['machine_failure'].mean()

C:\Users\taham\AppData\Local\Temp\ipykernel_28848\3838300079.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(pd.cut(df['tool_wear_min'], bins=10))['machine_failure'].mean()


tool_wear_min
(-0.253, 25.3]    0.025478
(25.3, 50.6]      0.017544
(50.6, 75.9]      0.022807
(75.9, 101.2]     0.025445
(101.2, 126.5]    0.022688
(126.5, 151.8]    0.019315
(151.8, 177.1]    0.021830
(177.1, 202.4]    0.038698
(202.4, 227.7]    0.151713
(227.7, 253.0]    0.338983
Name: machine_failure, dtype: float64

In [14]:
df['temp_gap'] = df['process_temperature_k'] - df['air_temperature_k']
df.groupby(pd.cut(df['temp_gap'], bins=8))['machine_failure'].mean()

C:\Users\taham\AppData\Local\Temp\ipykernel_28848\1607287993.py:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby(pd.cut(df['temp_gap'], bins=8))['machine_failure'].mean()


temp_gap
(7.595, 8.162]      0.146965
(8.162, 8.725]      0.133333
(8.725, 9.287]      0.025226
(9.287, 9.85]       0.022913
(9.85, 10.413]      0.022879
(10.413, 10.975]    0.024924
(10.975, 11.538]    0.022222
(11.538, 12.1]      0.011194
Name: machine_failure, dtype: float64

In [15]:
df['high_wear'] = (df['tool_wear_min'] > 200).astype(int)
df['low_temp_gap'] = (df['temp_gap'] < 8.7).astype(int)

df[['tool_wear_min', 'high_wear', 'temp_gap', 'low_temp_gap', 'machine_failure']].head(10)


,tool_wear_min,high_wear,temp_gap,low_temp_gap,machine_failure
0,50,0,10.4,0,0
1,55,0,10.4,0,0
2,58,0,10.4,0,0
3,63,0,10.5,0,0
4,65,0,10.4,0,0
5,68,0,10.4,0,0
6,70,0,10.4,0,0
7,73,0,10.5,0,0
8,75,0,10.4,0,0
9,77,0,10.3,0,0


In [16]:
df[df['high_wear'] == 1][['tool_wear_min', 'high_wear', 'machine_failure']].head(10)


,tool_wear_min,high_wear,machine_failure
56,202,1,0
57,204,1,0
58,206,1,0
59,208,1,1
138,203,1,0
139,206,1,0
140,211,1,0
141,214,1,0
142,216,1,1
143,218,1,1


In [17]:
df[df['low_temp_gap'] == 1][['temp_gap', 'low_temp_gap', 'machine_failure']].head(10)

,temp_gap,low_temp_gap,machine_failure
2906,8.7,1,0
2958,8.7,1,0
2959,8.7,1,0
2966,8.7,1,0
2973,8.7,1,0
3231,8.6,1,0
3232,8.6,1,0
3233,8.6,1,0
3234,8.6,1,0
3235,8.6,1,0


In [18]:
df[df['low_temp_gap'] == 1]['machine_failure'].mean()

0.14144736842105263

In [19]:
# One-hot encode 'type'
df = pd.get_dummies(df, columns=['type'], prefix='type')

# Drop leakage and identifier columns
df_model = df.drop(columns=['udi', 'product_id', 'loaded_at', 'twf', 'hdf', 'pwf', 'osf', 'rnf'])

df_model.head()

,air_temperature_k,process_temperature_k,rotational_speed_rpm,torque_nm,tool_wear_min,machine_failure,temp_gap,high_wear,low_temp_gap,type_H,type_L,type_M
0,298.8,309.2,1306,54.5,50,0,10.4,0,0,True,False,False
1,298.9,309.3,1632,32.5,55,0,10.4,0,0,False,False,True
2,298.9,309.3,1375,42.7,58,0,10.4,0,0,True,False,False
3,298.8,309.3,1450,44.8,63,0,10.5,0,0,False,True,False
4,298.9,309.3,1581,30.7,65,0,10.4,0,0,False,False,True


In [20]:
df_model.to_csv('../data/processed_features.csv', index=False)
print("Saved:", df_model.shape)

Saved: (10000, 12)


In [21]:
from sklearn.model_selection import train_test_split

# Separate features (X) from target (y)
X = df_model.drop(columns=['machine_failure'])
y = df_model['machine_failure']

# Split: 80% train, 20% test, stratified to preserve the 3.39% failure rate in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)
print("Train failure rate:", y_train.mean())
print("Test failure rate:", y_test.mean())

Train shape: (8000, 11)
Test shape: (2000, 11)
Train failure rate: 0.033875
Test failure rate: 0.034


In [22]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

# Train baseline model
log_reg = LogisticRegression(max_iter=1000, class_weight='balanced')
log_reg.fit(X_train, y_train)

# Predict on the unseen test set
y_pred_log = log_reg.predict(X_test)

print(classification_report(y_test, y_pred_log))

              precision    recall  f1-score   support

           0       1.00      0.88      0.93      1932
           1       0.20      0.88      0.33        68

    accuracy                           0.88      2000
   macro avg       0.60      0.88      0.63      2000
weighted avg       0.97      0.88      0.91      2000



In [23]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report

# scale_pos_weight tells XGBoost how imbalanced the classes are
scale = (y_train == 0).sum() / (y_train == 1).sum()

xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    scale_pos_weight=scale,
    eval_metric='logloss',
    random_state=42
)

xgb_model.fit(X_train, y_train)

y_pred_xgb = xgb_model.predict(X_test)

print(classification_report(y_test, y_pred_xgb))

              precision    recall  f1-score   support

           0       0.99      0.98      0.99      1932
           1       0.58      0.82      0.68        68

    accuracy                           0.97      2000
   macro avg       0.79      0.90      0.83      2000
weighted avg       0.98      0.97      0.98      2000



In [24]:
import joblib
import os

os.makedirs('../models', exist_ok=True)
joblib.dump(xgb_model, '../models/xgb_model.pkl')
print("Model saved to ../models/xgb_model.pkl")

Model saved to ../models/xgb_model.pkl


In [25]:
import pandas as pd

importance = pd.Series(xgb_model.feature_importances_, index=X_train.columns)
importance.sort_values(ascending=False)

rotational_speed_rpm     0.317188
tool_wear_min            0.228256
torque_nm                0.227690
temp_gap                 0.080604
air_temperature_k        0.040395
type_M                   0.039304
process_temperature_k    0.033618
type_H                   0.016741
type_L                   0.016205
high_wear                0.000000
low_temp_gap             0.000000
dtype: float32

In [26]:
import mlflow
import mlflow.xgboost

mlflow.set_experiment("predictive-maintenance")

with mlflow.start_run(run_name="xgboost_v1"):
    mlflow.log_param("n_estimators", 200)
    mlflow.log_param("max_depth", 4)
    mlflow.log_param("learning_rate", 0.1)
    mlflow.log_param("scale_pos_weight", scale)

    xgb_model.fit(X_train, y_train)
    y_pred = xgb_model.predict(X_test)

    from sklearn.metrics import precision_score, recall_score, f1_score
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    mlflow.log_metric("precision", precision)
    mlflow.log_metric("recall", recall)
    mlflow.log_metric("f1_score", f1)

    mlflow.xgboost.log_model(xgb_model, "model")

    print(f"Logged run — precision: {precision:.3f}, recall: {recall:.3f}, f1: {f1:.3f}")

c:\Users\taham\predictive-maintenance\.venv\Lib\site-packages\xgboost\sklearn.py:1183: UserWarning: [21:23:52] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\c_api\c_api.cc:1553: Saving model in the UBJSON format as default.  You can use a file extension: `json` or `ubj` to choose between formats.
  self.get_booster().save_model(fname)


Logged run — precision: 0.583, recall: 0.824, f1: 0.683
